In [1]:

import sys
import subprocess
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU memory (GB):", round(gpu.total_memory / (1024**3), 2))

print("\nNVIDIA GPU information:")
subprocess.run(["nvidia-smi"], check=False)


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56

NVIDIA GPU information:


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [2]:

%pip install -q -U \
    "transformers" \
    "accelerate" \
    "peft" \
    "trl" \
    "datasets" \
    "bitsandbytes" \
    "safetensors"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.2/948.2 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 15.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.


In [3]:

import importlib.metadata as metadata
import torch

packages = [
    "transformers",
    "accelerate",
    "peft",
    "trl",
    "datasets",
    "bitsandbytes",
    "safetensors"
]

print("=== LLM DEPENDENCY VERIFICATION ===")

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: NOT INSTALLED")

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    free_memory, total_memory = torch.cuda.mem_get_info()
    print(
        "Free GPU memory (GB):",
        round(free_memory / (1024**3), 2)
    )
    print(
        "Total GPU memory (GB):",
        round(total_memory / (1024**3), 2)
    )


=== LLM DEPENDENCY VERIFICATION ===
transformers: 5.20.1
accelerate: 1.15.0
peft: 0.21.2
trl: 1.16.0
datasets: 5.1.0
bitsandbytes: 0.50.2
safetensors: 0.8.0

CUDA available: True
Free GPU memory (GB): 14.46
Total GPU memory (GB): 14.56


In [4]:

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=False
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Qwen2.5-Coder-7B-Instruct in 4-bit mode...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    trust_remote_code=False
)

model.eval()

print("\n=== MODEL LOAD VERIFICATION ===")
print("Model:", MODEL_ID)
print("Model type:", type(model).__name__)
print("Device:", model.device)

allocated = torch.cuda.memory_allocated() / (1024**3)
reserved = torch.cuda.memory_reserved() / (1024**3)
free, total = torch.cuda.mem_get_info()

print(f"Allocated GPU memory: {allocated:.2f} GB")
print(f"Reserved GPU memory: {reserved:.2f} GB")
print(f"Free GPU memory: {free / (1024**3):.2f} GB")

print("\nMODEL LOAD SUCCESS")


Loading tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


=== MODEL LOAD VERIFICATION ===
Model: Qwen/Qwen2.5-Coder-7B-Instruct
Model type: Qwen2ForCausalLM
Device: cuda:0
Allocated GPU memory: 5.18 GB
Reserved GPU memory: 5.33 GB
Free GPU memory: 9.11 GB

MODEL LOAD SUCCESS


In [5]:

import torch
import time

# ============================================================
# MILESTONE 28D - QWEN BASELINE INFERENCE
# Research: Code Cleanliness & Maintainability
# Model: Qwen2.5-Coder-7B-Instruct
# Purpose: Evidence-grounded maintainability explanation
# ============================================================

# 1. System instructions
system_prompt = """
You are a Java software maintainability explanation assistant.

You receive findings from a deterministic Java static analyzer.

Your responsibilities:
- Explain only the supplied maintainability findings.
- Explain why the findings may affect maintainability.
- Recommend appropriate refactoring directions.
- Ground technical claims in the provided evidence.
- Distinguish measured evidence from possible consequences.
- Do not invent metrics, code smells, dependencies, or facts.
- Do not claim refactoring is behavior-preserving without tests.

The Java analyzer performs detection, grouping, and prioritization.
You provide explanations and refactoring-oriented guidance only.
"""

# 2. Controlled example of Java analyzer evidence
analysis_evidence = """
REPOSITORY ANALYSIS EXAMPLE

Affected file: src/main/java/example/OrderService.java
Affected class: OrderService
Affected method: processOrder()

Detected concerns:

1. LONG_METHOD
   - Measured NCSS: 64
   - Configured NCSS threshold: 30

2. HIGH_COMPLEXITY
   - Measured cyclomatic complexity: 18
   - Configured complexity threshold: 10

Related issue group:
- Both concerns affect processOrder().
- Grouping reason: same method context.

Maintainability priority:
- High review priority assigned by the analyzer.
- Priority must not be recalculated by the LLM.

Relevant Java code excerpt:

public void processOrder(Order order) {
    if (order == null) {
        return;
    }

    if (order.isValid()) {
        calculateTotal(order);
        applyDiscount(order);
        updateInventory(order);
        sendConfirmation(order);
    } else {
        rejectOrder(order);
    }
}

IMPORTANT:
The metric values are supplied analyzer evidence for a
controlled demonstration.

The code excerpt is incomplete. It is not sufficient
to independently verify the supplied NCSS and
cyclomatic complexity measurements.
"""

# 3. User prompt
user_prompt = f"""
Explain the following Java maintainability triage result.

{analysis_evidence}

Provide these five sections:

1. Maintainability Concern
2. Evidence-Based Explanation
3. Why It Matters
4. Suggested Refactoring Direction
5. Limitations and Verification

Requirements:
- Be technically accurate.
- Use only the supplied evidence.
- Explain the relationship between the two concerns.
- Do not invent additional maintainability findings.
- Do not independently change the priority.
- Clearly acknowledge the incomplete code excerpt.
- Recommend verification before applying refactoring.
"""

# 4. Prepare messages for the Qwen chat model
messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

# 5. Tokenize messages using Qwen's chat template
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
)

# 6. Move inputs to the GPU
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

input_length = inputs["input_ids"].shape[-1]

print("=== QWEN BASELINE INFERENCE ===")
print("Model:", MODEL_ID)
print("Input tokens:", input_length)
print("Device:", model.device)

# 7. Prepare model and GPU measurement
model.eval()

torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()

# 8. Generate explanation
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=450,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

end_time = time.perf_counter()

# 9. Extract generated text only
generated_tokens = outputs[0][input_length:]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

# 10. Display generated explanation
print("\n" + "=" * 60)
print("GENERATED MAINTAINABILITY EXPLANATION")
print("=" * 60)

print(response)

# 11. Calculate inference metrics
elapsed_time = end_time - start_time

output_token_count = generated_tokens.shape[-1]

tokens_per_second = (
    output_token_count / elapsed_time
    if elapsed_time > 0
    else 0
)

peak_gpu_memory = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
)

free_gpu_memory, total_gpu_memory = (
    torch.cuda.mem_get_info()
)

# 12. Display performance metrics
print("\n" + "=" * 60)
print("BASELINE INFERENCE PERFORMANCE")
print("=" * 60)

print("Input tokens:", input_length)
print("Output tokens:", output_token_count)

print(
    "Generation time (seconds):",
    round(elapsed_time, 2)
)

print(
    "Generation speed (tokens/sec):",
    round(tokens_per_second, 2)
)

print(
    "Peak GPU memory (GB):",
    round(peak_gpu_memory, 2)
)

print(
    "Remaining free GPU memory (GB):",
    round(free_gpu_memory / (1024 ** 3), 2)
)

print("\nBASELINE INFERENCE COMPLETE")


=== QWEN BASELINE INFERENCE ===
Model: Qwen/Qwen2.5-Coder-7B-Instruct
Input tokens: 477
Device: cuda:0

GENERATED MAINTAINABILITY EXPLANATION
### 1. Maintainability Concerns

**LONG_METHOD**
- **Measured NCSS:** 64
- **Configured NCSS Threshold:** 30

**HIGH_COMPLEXITY**
- **Measured Cyclomatic Complexity:** 18
- **Configured Complexity Threshold:** 10

### 2. Evidence-Based Explanation

#### LONG_METHOD
The `processOrder()` method has a Non-Comment Source Statements (NCSS) count of 64, which exceeds the configured threshold of 30. This indicates that the method contains a large number of statements, making it difficult to understand and maintain.

#### HIGH_COMPLEXITY
The method has a cyclomatic complexity of 18, which is higher than the configured threshold of 10. Cyclomatic complexity measures the number of linearly independent paths through the source code, indicating how complex the control flow is within the method.

### 3. Why It Matters

Both issues—long methods and high comple

In [6]:

import json
from google.colab import files

uploaded = files.upload()

filename = next(iter(uploaded))

report = json.loads(uploaded[filename].decode("utf-8"))

assert report["schema_version"] == "1.1"
assert report["failed_files"] == 0
assert report["finding_count"] > 0

print("Files parsed:", report["parsed_files"])
print("Findings:", report["finding_count"])
print("Issue groups:", report["group_count"])
print("Schema:", report["schema_version"])


Saving analyzer-self-analysis-2026-10-18.json to analyzer-self-analysis-2026-10-18.json
Files parsed: 57
Findings: 44
Issue groups: 30
Schema: 1.1


In [7]:

group = report["ranked_groups"][0]

# Keep all findings in this issue group, but avoid
# duplicating identical source excerpts.
findings = []

for index, finding in enumerate(group["findings"]):
    item = {
        "concern": finding["concern"],
        "symbol": finding["symbol"],
        "rule_id": finding["rule_id"],
        "description": finding["description"],
        "start_line": finding["start_line"],
        "end_line": finding["end_line"],
        "evidence": finding["evidence"],
        "context_status": finding["context_status"],
    }

    if index == 0:
        item["code_context"] = finding.get("code_context")

    findings.append(item)

evidence = {
    "group_id": group["group_id"],
    "rank": group["rank"],
    "exceedance_score": group["exceedance_score"],
    "ranking_explanation": group["ranking_explanation"],
    "findings": findings,
}

print("Selected priority:", evidence["rank"])
print("Concerns:", [f["concern"] for f in findings])
print("Source context:", findings[0]["context_status"])


Selected priority: 1
Concerns: ['HIGH_COMPLEXITY', 'LONG_METHOD']
Source context: AVAILABLE_TRUNCATED_SOURCE_CONTEXT


In [8]:

import json
import torch

assert "model" in globals(), "Qwen model must be loaded first"
assert "tokenizer" in globals(), "Tokenizer must be loaded first"
assert "evidence" in globals(), "Run the evidence preparation cell first"

system_prompt = """
You are a Java maintainability explanation assistant.

Use only the supplied deterministic analyzer evidence.

Requirements:
- Do not invent metrics, thresholds, source code or dependencies.
- Treat detection results as preliminary.
- Do not change the supplied priority ranking.
- Identify the method and concerns accurately.
- Explain what the recorded metrics indicate.
- Suggest only conditional, evidence-supported improvements.
- Acknowledge truncated source context.
- Do not claim behavioral preservation without testing.
- Do not discuss unrelated quality components.
"""

user_prompt = (
    "Explain these real Java maintainability findings.\n\n"
    + json.dumps(evidence, ensure_ascii=False, indent=2)
    + "\n\nProvide:\n"
    "1. Identified method and concerns\n"
    "2. Recorded metric evidence\n"
    "3. Maintainability explanation\n"
    "4. Possible improvements\n"
    "5. Limitations and required verification"
)

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

# Transformers 5.x may return BatchEncoding.
# Explicitly extract input_ids as a tensor.
encoded = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt"
)

if isinstance(encoded, torch.Tensor):
    input_ids = encoded
elif isinstance(encoded, dict) or hasattr(encoded, "input_ids"):
    input_ids = encoded["input_ids"]
else:
    raise TypeError(f"Unexpected tokenizer result: {type(encoded)}")

input_ids = input_ids.to(model.device)

print("=== REAL ANALYZER EVIDENCE INFERENCE ===")
print("Method:", evidence["findings"][0]["symbol"])
print("Concerns:", [f["concern"] for f in evidence["findings"]])
print("Input tokens:", input_ids.shape[-1])
print("Device:", input_ids.device)

model.eval()

with torch.inference_mode():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=600,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

generated_ids = outputs[0][input_ids.shape[-1]:]

real_explanation = tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)

print("\n=== REAL ANALYZER EVIDENCE → QWEN ===\n")
print(real_explanation)


=== REAL ANALYZER EVIDENCE INFERENCE ===
Method: PoorNamingDetector.detect(RepositoryParseResult, NamingDetectionConfig)
Concerns: ['HIGH_COMPLEXITY', 'LONG_METHOD']
Input tokens: 2531
Device: cuda:0

=== REAL ANALYZER EVIDENCE → QWEN ===

### Identified Method and Concerns
The identified method is `PoorNamingDetector.detect(RepositoryParseResult, NamingDetectionConfig)` from the class `PoorNamingDetector`. The concerns detected are:

1. **High Complexity**: The method has a cyclomatic complexity of 23, which exceeds the configured preliminary decision boundary of 10.
2. **Long Method**: The method has an NCSS (Number of Cyclomatic Statements) of 46, which exceeds the configured preliminary decision boundary of 30.

### Recorded Metric Evidence
- **Cyclomatic Complexity**:
  - Configured Max Complexity: 10
  - Actual Cyclomatic Complexity: 23
  - Measurement Type: preliminary_ast_cyclomatic

- **NCSS**:
  - Configured Max NCSS: 30
  - Actual NCSS: 46
  - Measurement Type: preliminary_a

In [9]:
from google.colab import files

files.upload()  # Choose colab_qlora_pilot.py

Saving colab_qlora_pilot.py to colab_qlora_pilot.py


{'colab_qlora_pilot.py': b'"""Colab T4 QLoRA *smoke test* (NOT scientific model comparison).\nUpload train_pilot.jsonl and holdout_pilot.jsonl via the Colab upload prompt.\nRuns in one Colab code cell via: exec(open(\'colab_qlora_pilot.py\').read())\nNeeds GPU and current compatible Transformers/PEFT installed.\n"""\nimport json, torch\nfrom pathlib import Path\nfrom google.colab import files\nfrom transformers import (AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig,\n                          TrainingArguments,Trainer,DataCollatorForLanguageModeling)\nfrom peft import LoraConfig,prepare_model_for_kbit_training,get_peft_model\nprint(\'Upload: train_pilot.jsonl and holdout_pilot.jsonl\')\nup=files.upload()\ntrain=json.loads(up[\'train_pilot.jsonl\'].decode(\'utf-8\'))\nhold=json.loads(up[\'holdout_pilot.jsonl\'].decode(\'utf-8\'))\nassert train[\'source_project\']!=hold[\'source_project\'], \'PROJECT LEAKAGE\'\nassert train[\'status\']==\'USER_APPROVED_SOURCE_REVIEWED_NOT_BEHAVIOR

In [12]:
from google.colab import files

uploaded_files = files.upload()

print("Uploaded files:", list(uploaded_files.keys()))

Saving holdout_pilot.jsonl to holdout_pilot (2).jsonl
Saving train_pilot.jsonl to train_pilot.jsonl
Uploaded files: ['holdout_pilot (2).jsonl', 'train_pilot.jsonl']


In [13]:

import json
from pathlib import Path

train_path = Path("/content/train_pilot.jsonl")
holdout_path = Path("/content/holdout_pilot.jsonl")

assert train_path.exists(), "Training file missing"
assert holdout_path.exists(), "Holdout file missing"

train_record = json.loads(train_path.read_text(encoding="utf-8"))
holdout_record = json.loads(holdout_path.read_text(encoding="utf-8"))

print("Training:", train_record["case_id"])
print("Holdout:", holdout_record["case_id"])
print("Different projects:",
      train_record["source_project"] != holdout_record["source_project"])

print("Both files verified successfully.")


Training: Extract_Method_1
Holdout: Extract_Method_7
Different projects: True
Both files verified successfully.


In [14]:

from pathlib import Path
from unittest.mock import patch

script = Path("/content/colab_qlora_pilot.py").read_text(encoding="utf-8")

with patch("google.colab.files.upload", return_value={
    "train_pilot.jsonl": train_path.read_bytes(),
    "holdout_pilot.jsonl": holdout_path.read_bytes()
}):
    exec(
        compile(script, "/content/colab_qlora_pilot.py", "exec"),
        globals()
    )


Upload: train_pilot.jsonl and holdout_pilot.jsonl


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

trainable params: 20,185,088 || all params: 7,635,801,600 || trainable%: 0.2643


Step,Training Loss
1,3.698466
2,3.504061
3,3.330471
4,3.201380
5,3.108519


PILOT COMPLETE: 5 update steps on ONE training record; NO scientific performance claim
Adapter: /content/qlora_pilot_adapter (download/Drive-copy before session disconnect)
Holdout was NOT used in training: Extract_Method_7


In [15]:

import json
import shutil
from pathlib import Path
from google.colab import files

adapter_dir = Path("/content/qlora_pilot_adapter")
assert adapter_dir.exists(), "Adapter directory missing"

output_dir = Path("/content/qlora_pilot_backup")
output_dir.mkdir(exist_ok=True)

shutil.copytree(
    adapter_dir,
    output_dir / "adapter",
    dirs_exist_ok=True
)

history = trainer.state.log_history

(output_dir / "training_history.json").write_text(
    json.dumps(history, indent=2),
    encoding="utf-8"
)

shutil.copy(
    "/content/train_pilot.jsonl",
    output_dir / "train_pilot.jsonl"
)

shutil.copy(
    "/content/holdout_pilot.jsonl",
    output_dir / "holdout_pilot.jsonl"
)

archive = shutil.make_archive(
    "/content/qlora_pilot_backup",
    "zip",
    root_dir="/content",
    base_dir="qlora_pilot_backup"
)

print("Adapter and training evidence saved:", archive)

files.download(archive)


Adapter and training evidence saved: /content/qlora_pilot_backup.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
from google.colab import files
files.upload()

Saving milestone_29e_colab_compare.py to milestone_29e_colab_compare.py


{'milestone_29e_colab_compare.py': b'"""Milestone 29E: base vs QLoRA adapter on a project-disjoint ONE-case holdout.\nColab T4. Upload qlora_pilot_backup.zip produced by Milestone 29D.\nThis is a reproducibility/qualitative pilot, NOT research-grade evaluation.\n"""\nimport gc\nimport json\nimport re\nimport shutil\nimport tempfile\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport torch\nfrom google.colab import files\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig\nfrom peft import PeftModel\n\nMODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"\nOUT = Path(\'/content/milestone_29e_results\')\nOUT.mkdir(parents=True, exist_ok=True)\n\nbackup = Path(\'/content/qlora_pilot_backup.zip\')\nif not backup.is_file():\n    print(\'Upload the existing qlora_pilot_backup.zip (NOT individual JSONL files).\')\n    uploaded = files.upload()\n    matches = [Path(\'/content\') / n for n in uploaded if n.endswith(\'.zip\')]\n    ass

In [17]:
%run /content/milestone_29e_colab_compare.py

Holdout project: hibernate-orm
Training project: checkstyle
Adapter path: /content/29e_qlora_backup/qlora_pilot_backup/adapter


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Holdout prompt tokens: 1243

===== BASE QWEN =====
The `DefaultRefreshEventListener.refresh` method in Hibernate ORM has 31 non-comment, non-blank statements (NCSS), which exceeds the configured maximum of 30 NCSS allowed by the rule `METHOD_NCSS_BOUNDARY`. This indicates that the method is quite long and complex.

### Analysis:
1. **Method Length**: The method contains multiple nested conditional blocks and loops, making it difficult to understand at a glance.
   - It checks whether the object is transient and handles both cases separately.
   - It cascades refresh operations before refreshing the current entity.
   - It evicts entities and collections, and performs various checks and updates.

2. **Complexity**:
   - The method involves several steps including checking object status, cascading operations, and updating persistence context.
   - There are multiple points where exceptions can be thrown, increasing the complexity of error handling.

3. **Readability**:
   - The method la

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>